# CAD: is CHASE a fixed-joint recentring? (3 seeds)
Arms, all CTR-GCN with the released CAD config:
- **chase**: released CHASE (CLB + MPMMD), unchanged
- **fixed**: origin := joint 4 of person slot 0 at frame 3 (the point CHASE selects in 761/761 test clips), no CLB, no MPMMD
- **vanilla**: no shift

Same seeds (1, 2, 3), same protocol as the paper (best test accuracy, evaluated every 3 epochs). Results are appended after **each** run to `cad_fixed_origin_results.csv`, so a timeout keeps the finished runs. Re-running the notebook with the previous output attached skips finished runs only if you copy that csv into `/kaggle/working` first.

**Settings:** GPU T4, Internet On, **Add Input → your CAD dataset** (from the earlier fix). Then **Save Version → Save & Run All**.

## 1. Setup

In [ ]:
import os, sys, re, glob, json, time, zipfile, tarfile, pickle, shutil, subprocess
def sh(cmd, cwd=None):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

WORK = '/kaggle/working'
REPO = WORK + '/CHASE/NeurIPS24'
if not os.path.isdir(REPO):
    sh('git clone -q --depth 1 https://github.com/Necolizer/CHASE.git', cwd=WORK)
sh('pip install -q dotmap einops pyyaml')
import torch, yaml
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU: Session options -> Accelerator -> GPU T4'


## 2. CAD data

In [ ]:
# CAD skeletons from the attached Kaggle dataset (Add Input -> your cad-composer dataset)
SRC = '/kaggle/input'
DATA_ROOT = '/tmp/cad'
os.makedirs(DATA_ROOT, exist_ok=True)
hits = glob.glob(DATA_ROOT + '/**/tracks_normalized.pkl', recursive=True)
if not hits:
    src_hits = glob.glob(SRC + '/**/tracks_normalized.pkl', recursive=True)
    if src_hits:                                   # extracted, read-only -> copy skeletons
        dst = os.path.join(DATA_ROOT, 'collective')
        shutil.copytree(os.path.dirname(src_hits[0]), dst, ignore=shutil.ignore_patterns('videos'))
    else:                                          # still an archive -> unpack without videos
        arcs = [p for p in glob.glob(SRC + '/**/*', recursive=True)
                if os.path.isfile(p) and (zipfile.is_zipfile(p) or tarfile.is_tarfile(p))]
        assert arcs, 'Attach the CAD dataset via Add Input'
        a = arcs[0]
        if zipfile.is_zipfile(a):
            with zipfile.ZipFile(a) as z:
                z.extractall(DATA_ROOT, [m for m in z.namelist() if 'videos' not in m.split('/')])
        else:
            with tarfile.open(a) as t:
                t.extractall(DATA_ROOT, [m for m in t.getmembers() if 'videos' not in m.name.split('/')])
    hits = glob.glob(DATA_ROOT + '/**/tracks_normalized.pkl', recursive=True)
assert hits, 'tracks_normalized.pkl not found'
CAD_DIR = os.path.dirname(hits[0])
print('CAD_DIR =', CAD_DIR, sorted(os.listdir(CAD_DIR)))


## 3. Fixed-origin wrapper

In [ ]:
FX = r'''"""
fixed_origin.py -- baselines for the CAD "is CHASE a fixed-joint recentring?" test.
Copy to CHASE/NeurIPS24/model/Wrapper/fixed_origin.py.

mode='joint' : origin := x[:, :, t, v, m] for every sample (the point the released
               CHASE checkpoint selects in 761/761 CAD test clips: t=3, v=4, m=0).
               Padding is handled exactly as in CHASWrapper (x - p for all points),
               so the ONLY difference from CHASE is that the origin is fixed, not learned.
mode='none'  : no shift (vanilla backbone), same wrapper so eval returns logits.
"""
import torch.nn as nn


class FixedOriginWrapper(nn.Module):
    def __init__(self, backbone, mode="joint", t=3, v=4, m=0, **unused):
        super().__init__()
        assert mode in ("joint", "none")
        self.backbone, self.mode, self.t, self.v, self.m = backbone, mode, t, v, m

    def forward(self, x):
        if self.mode == "joint":
            p = x[:, :, self.t, self.v, self.m]                     # (N, C)
            x = x - p[:, :, None, None, None]
        out = self.backbone(x)
        if not self.training and isinstance(out, tuple):
            return out[0]
        return out
'''
open(REPO + '/model/Wrapper/fixed_origin.py', 'w').write(FX)
init = REPO + '/model/Wrapper/__init__.py'
if 'fixed_origin' not in open(init).read():
    open(init, 'a').write('\nfrom . import fixed_origin\n')
print('wrapper written and registered')


## 4. Configs

In [ ]:
base = yaml.safe_load(open(REPO + '/config/CTRGCN/cad/cad_chase.yaml'))
for k in ('train_feeder_args', 'test_feeder_args'):
    base[k]['dataset_dir'] = CAD_DIR
# Memory: the paper's batch 256 (6 persons) needs > 16 GB on one GPU.
NGPU = torch.cuda.device_count()
if NGPU >= 2:      # GPU T4 x2: keep batch 256 / lr 0.1, split over 2 GPUs (DataParallel)
    base.update(device=[0, 1], cuda_visible_device='0,1')
else:              # single GPU: halve batch, scale lr linearly (same for every arm)
    base.update(device=[0], cuda_visible_device='0', batch_size=128, test_batch_size=128,
                base_lr=base['base_lr'] * 0.5)
base.update(num_worker=4, run_mode='train')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('GPUs:', NGPU, '| batch', base['batch_size'], '| lr', base['base_lr'])

def arm_cfg(arm):
    c = json.loads(json.dumps(base))
    if arm == 'chase':                         # released CHASE, unchanged
        pass
    elif arm in ('fixed', 'vanilla'):
        c['wrapper'] = 'model.Wrapper.fixed_origin.FixedOriginWrapper'
        c['wrapper_args'] = dict(mode='joint' if arm == 'fixed' else 'none', t=3, v=4, m=0)
        c['loss'] = 'LSCE_GROUP'               # no MPMMD: nothing in these arms can minimise it
        c['loss_args'] = dict(smoothing=base['loss_args']['smoothing'],
                              temperature=base['loss_args']['temperature'])
    path = f'{REPO}/cad_{arm}.yaml'
    yaml.safe_dump(c, open(path, 'w'))
    return path

for arm in ('chase', 'fixed', 'vanilla'):
    print(arm, '->', arm_cfg(arm))


## 5. Train (9 runs)

In [ ]:
# Order: paired CHASE/fixed runs first, vanilla last (so a timeout loses the least important arm).
SEEDS = [1, 2, 3]
PLAN = [(a, s) for s in SEEDS for a in ('fixed', 'chase')] + [('vanilla', s) for s in SEEDS]
RES = WORK + '/cad_fixed_origin_results.csv'
done = set()
if os.path.exists(RES):
    for line in open(RES).read().splitlines()[1:]:
        a, s = line.split(',')[:2]; done.add((a, int(s)))
else:
    open(RES, 'w').write('arm,seed,best_acc,last_acc,minutes\n')

os.makedirs(WORK + '/runs', exist_ok=True)
pat = re.compile(r'top1: ([\d.]+)%, best_acc: ([\d.]+)%')
for arm, seed in PLAN:
    if (arm, seed) in done:
        continue
    wd = f'{WORK}/runs/cad_{arm}_s{seed}'
    shutil.rmtree(wd, ignore_errors=True)
    t0 = time.time()
    sh(f'python main_group.py --config cad_{arm}.yaml --seed {seed} --work_dir {wd} > {wd}.out 2>&1 || (tail -40 {wd}.out; false)', cwd=REPO)
    m = pat.findall(open(wd + '/log.txt').read())
    last, best = float(m[-1][0]), float(m[-1][1])
    mins = (time.time() - t0) / 60
    open(RES, 'a').write(f'{arm},{seed},{best},{last},{mins:.1f}\n')
    print(f'>>> {arm} seed {seed}: best {best:.2f}  last {last:.2f}  ({mins:.1f} min)', flush=True)


## 6. Summary

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
df = pd.read_csv(RES); print(df.to_string(index=False))
g = df.groupby('arm')[['best_acc', 'last_acc']].agg(['mean', 'std', 'count']).round(2)
print('\n', g)
p = df.pivot(index='seed', columns='arm', values='best_acc')
if {'fixed', 'chase'} <= set(p.columns):
    d = (p['fixed'] - p['chase']).dropna()
    print('\npaired best_acc difference fixed - chase per seed:', d.round(2).tolist())
    if len(d) >= 2:
        t = stats.ttest_rel(p.loc[d.index, 'fixed'], p.loc[d.index, 'chase'])
        print(f'mean diff {d.mean():.2f}  sd {d.std(ddof=1):.2f}  paired t p = {t.pvalue:.3f}')
json.dump(dict(runs=df.to_dict('records')), open(WORK + '/cad_fixed_origin_summary.json', 'w'), indent=2)
print('\nSend back: /kaggle/working/cad_fixed_origin_results.csv')
